<a href="https://colab.research.google.com/github/zhengpohung/1d-tokenizer/blob/text_guide/text_guided_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision numpy pandas pillow
!pip install sionna-gpu  # Sionna 需要 CUDA 環境
!pip install transformers accelerate sentencepiece protobuf
!pip install omegaconf  # 讀取配置檔案
!pip install git+https://www.google.com/search?q=https://github.com/bytedance/1d-tokenizer.git  # 安裝 1d-tokenizer

ERROR: Could not find a version that satisfies the requirement sionna-gpu (from versions: none)
ERROR: No matching distribution found for sionna-gpu
  Cloning https://www.google.com/search?q=https://github.com/bytedance/1d-tokenizer.git to /tmp/pip-req-build-hun9n68w
  Running command git clone --filter=blob:none --quiet 'https://www.google.com/search?q=https://github.com/bytedance/1d-tokenizer.git' /tmp/pip-req-build-hun9n68w
  fatal: https://www.google.com/search?q=https://github.com/bytedance/1d-tokenizer.git/info/refs not valid: is this a git repository?
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet 'https://www.google.com/search?q=https://github.com/bytedance/1d-tokenizer.git' /tmp/pip-req-build-hun9n68w did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet 'https

In [ ]:
!pip install torchmetrics==0.11.4  # 確保版本相容性
!pip install lpips  # 感知損失庫

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TokCom_Simulation_Files'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"雲端硬碟根目錄設定為: {DRIVE_ROOT}")

In [ ]:
import torch import numpy as np import sionna import itertools from sionna.fec.polar import Polar5GEncoder, Polar5GDecoder from sionna.fec.crc import CRC from sionna.utils import ebnodb2no from sionna.channel import AWGN from sionna.mapping import Mapper, Demapper from transformers import AutoModelForCausalLM, AutoProcessor, CLIPTextModel, CLIPTokenizer, GenerationConfig from PIL import Image from torchvision import transforms from torch.utils.data import DataLoader, Dataset import time from omegaconf import OmegaConf

In [ ]:
try:
  from _1d_tokenizer.modeling.tatitok import TATiTok
  from _1d_tokenizer.modeling.maskgen import MaskGen
except ImportError:
# 如果 Colab 的環境路徑有問題，可能需要手動將 1d-tokenizer 複製到 /content
# 在 Colab 中，通常會將 repo 安裝到 site-packages 或使用相對路徑。
# 這裡我們依賴 pip install 的結果。如果仍失敗，請檢查安裝路徑。
  print("警告: 無法直接導入 1d-tokenizer 模組。請檢查安裝或路徑。")

In [ ]:
from torchmetrics.functional import peak_signal_noise_ratio as psnr_metric
from torchmetrics.image.lpips import LearnedPerceptualImagePatchSimilarity
from torchmetrics.text.clip_score import CLIPScore

In [ ]:
class ImageNetLoader(Dataset):
"""
自定義 ImageNet DataLoader 結構，用於從雲端硬碟讀取圖像。
實際應用中，您需要確保 {DRIVE_ROOT}/imagenet_val/ 包含標準的 ImageNet 分類結構。
"""
def init(self, root_dir, size=5):
  self.root_dir = root_dir
  self.transform = transforms.Compose([
  transforms.Resize((256, 256)),
  transforms.ToTensor(),
  ])

  # 模擬讀取 ImageNet 結構 (實際運行時，這會掃描整個資料夾)
  image_files = []
  # 這裡僅使用 Mock 模擬檔案列表
  for i in range(size):
      image_files.append(f"mock_image_{i}.jpg")

  self.image_files = image_files

def __len__(self):
    return len(self.image_files)

def __getitem__(self, idx):
    # 實際應用中：
    # img_path = os.path.join(self.root_dir, self.image_files[idx])
    # image = Image.open(img_path).convert('RGB')

    # 由於沒有 ImageNet 權限，這裡使用 Mock 圖片和 Caption
    image = Image.new('RGB', (256, 256), color=(idx*20 % 256, 100, 200))
    caption = f"A photo of a dog from ImageNet validation set, index {idx}."

    return image, caption

In [ ]:
class WirelessImageTransmissionSystem:
  def init(self, device='cuda', drive_root='/content/drive/MyDrive/TokCom_Simulation_Files'):
      self.device = device
      self.drive_root = drive_root
      print(f"初始化系統於 {device}...")

      # ---------------------------------------------------------
      # 1. 載入 AI 模型 (Molmo, CLIP, TA-TiTok, MaskGen)
      # ---------------------------------------------------------
      self._init_molmo()
      self._init_clip()
      self._init_tatitok_maskgen()
      self._init_metrics()

      # ---------------------------------------------------------
      # 2. 設置 5G PHY 層參數 (Sionna)
      # ---------------------------------------------------------
      # 論文設定：128 tokens, 8192 codebook -> 13 bits/token, 8 tokens/package, CRC11, N=256, 4-QAM
      self.tokens_per_image = 128
      self.bits_per_token = 13 # log2(8192)
      self.tokens_per_package = 8
      self.payload_bits = self.tokens_per_package * self.bits_per_token # 104
      self.crc_poly = "CRC11" # 5G NR Standard CRC11
      self.coder_n = 256 # 碼長

      # Sionna 組件
      self.crc_encoder = CRC(self.crc_poly)
      self.crc_decoder = CRC(self.crc_poly)

      # K (Polar 編碼輸入位元數) = Payload + CRC 長度
      self.k_polar = self.payload_bits + self.crc_encoder.crc_length # 104 + 11 = 115

      # 5G Polar Codec
      self.polar_encoder = Polar5GEncoder(k=self.k_polar, n=self.coder_n)
      self.polar_decoder = Polar5GDecoder(enc=self.polar_encoder, list_size=8)

      # 調變 (4-QAM)
      self.mapper = Mapper("qam", 4)
      self.demapper = Demapper("qam", 4, "app") # app = a posteriori probability (LLR)

      # 通道
      self.channel = AWGN()

  def _init_molmo(self):
      """初始化 Molmo-7B 用於圖像描述"""
      print("載入 Molmo-7B...")
      try:
          # 由於 Molmo 載入需要 transformers.GenerationConfig，我們從頂層導入 GenerationConfig
          self.molmo_processor = AutoProcessor.from_pretrained(
              "allenai/Molmo-7B-D-0924",
              trust_remote_code=True,
              torch_dtype=torch.float16, # 使用 float16 減少 VRAM 壓力
              device_map='auto'
          )
          self.molmo_model = AutoModelForCausalLM.from_pretrained(
              "allenai/Molmo-7B-D-0924",
              trust_remote_code=True,
              torch_dtype=torch.float16,
              device_map='auto'
          )
          self.use_molmo = True
          print("Molmo 載入成功。")
      except Exception as e:
          print(f"警告: Molmo 載入失敗 ({e})。將使用模擬描述。")
          self.use_molmo = False

  def _init_clip(self):
      """初始化 CLIP 用於文本編碼"""
      print("載入 CLIP...")
      self.clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
      # 為了節省 VRAM，我們將 CLIP 保持在 CPU 或與其他模型共享
      self.clip_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(self.device)

  def _init_tatitok_maskgen(self):
      """初始化 TA-TiTok 和 MaskGen"""
      print("載入 TA-TiTok 和 MaskGen...")

      # --- 雲端硬碟路徑設定 ---
      CONFIG_PATH_TA = os.path.join(self.drive_root, "configs", "tatitok_bl128_vq.yaml")
      CKPT_PATH_TA = os.path.join(self.drive_root, "checkpoints", "tatitok_bl128_vq.bin")

      CONFIG_PATH_MG = os.path.join(self.drive_root, "configs", "maskgen_vq_xl.yaml")
      CKPT_PATH_MG = os.path.join(self.drive_root, "checkpoints", "maskgen_vq_xl.bin")

      try:
          # --- TA-TiTok 載入 ---
          self.tatitok_config = OmegaConf.load(CONFIG_PATH_TA)
          self.tatitok = TATiTok(config=self.tatitok_config).to(self.device)
          self.tatitok.load_state_dict(torch.load(CKPT_PATH_TA, map_location=self.device)['state_dict'])
          self.tatitok.eval()
          print("TA-TiTok 載入成功。")

          # --- MaskGen 載入 ---
          self.maskgen_config = OmegaConf.load(CONFIG_PATH_MG)
          self.maskgen = MaskGen(config=self.maskgen_config).to(self.device)
          self.maskgen.load_state_dict(torch.load(CKPT_PATH_MG, map_location=self.device)['state_dict'])
          self.maskgen.eval()
          print("MaskGen 載入成功。")

          self.vocab_size = self.tatitok_config.model.codebook_size # 從 config 讀取
          self.mask_token_id = self.vocab_size # 通常為詞彙表大小

      except Exception as e:
          print(f"警告: TA-TiTok/MaskGen 載入失敗 ({e})。請檢查雲端硬碟路徑和檔案。")
          self.vocab_size = 8192
          self.mask_token_id = self.vocab_size
          pass

  def _init_metrics(self):
      """初始化評估指標"""
      self.lpips = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True).to(self.device)
      self.clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-large-patch14").to(self.device)
      print("Metric 模組已初始化。")

  def generate_caption(self, image):
      """使用 Molmo 生成描述 (Mocked 或 Real)"""
      if not self.use_molmo:
          return "A highly detailed image of a friendly golden retriever running on a sunny beach, captured in a photorealistic style."

      # 實際 Molmo 邏輯
      # 確保在 CPU 上載入的模型在推理時也使用 CPU 或正確的 device_map
      inputs = self.molmo_processor(
          images=image,
          text="Describe this image in detail.",
          return_tensors="pt"
      )

      output = self.molmo_model.generate(**inputs, max_new_tokens=100)
      generated_text = self.molmo_processor.tokenizer.decode(output[0, inputs.input_ids.size(1):], skip_special_tokens=True).strip()
      return generated_text

  def get_clip_embedding(self, caption):
      """將描述轉換為 77-token Embedding"""
      tokens = self.clip_tokenizer(
          caption,
          padding="max_length",
          max_length=77,
          truncation=True,
          return_tensors="pt"
      ).input_ids.to(self.device)

      with torch.no_grad():
          encoder_hidden_states = self.clip_encoder(tokens).last_hidden_state # [1, 77, 768]
      return encoder_hidden_states

  def int_to_bits(self, tokens):
      """Token IDs (int) 轉 bits"""
      masks = 1 << torch.arange(self.bits_per_token - 1, -1, -1).to(tokens.device)
      return ((tokens.unsqueeze(-1) & masks) > 0).int().flatten()

  def bits_to_int(self, bits):
      """bits 轉回 Token IDs"""
      bits = bits.view(-1, self.bits_per_token)
      masks = 1 << torch.arange(self.bits_per_token - 1, -1, -1).to(bits.device)
      return (bits * masks).sum(dim=1).int()

  @torch.no_grad()
  def transmission_channel(self, token_indices, snr_db):
      """模擬 5G PHY 通道傳輸"""

      num_packages = self.tokens_per_image // self.tokens_per_package
      token_packages = token_indices.view(num_packages, self.tokens_per_package)

      received_tokens_list = []
      package_error_flags = []

      for i in range(num_packages):
          package = token_packages[i]

          # --- 發射端 ---
          u = self.int_to_bits(package).float().unsqueeze(0) # [1, 104]
          c = self.crc_encoder(u) # [1, 115]
          x = self.polar_encoder(c) # [1, 256]
          x_sym = self.mapper(x)

          # --- 通道 (AWGN) ---
          ebn0_bit = snr_db + 10 * torch.log10(torch.tensor(self.k_polar / self.coder_n * 2))
          no = ebnodb2no(ebn0_bit, 2, self.k_polar/self.coder_n)

          y = self.channel([x_sym, no])

          # --- 接收端 ---
          llr = self.demapper([y, no])
          c_hat = self.polar_decoder(llr)
          u_hat, crc_valid = self.crc_decoder(c_hat)

          rec_tokens = self.bits_to_int(u_hat.squeeze(0))

          is_error = not crc_valid.item()

          if is_error:
              # CRC 檢查失敗，整個封包標記為 [MASK]
              masked_package = torch.full_like(rec_tokens, self.mask_token_id)
              received_tokens_list.append(masked_package)
              package_error_flags.extend([1] * self.tokens_per_package)
          else:
              received_tokens_list.append(rec_tokens)
              package_error_flags.extend([0] * self.tokens_per_package)

      # 重組
      final_tokens = torch.cat(received_tokens_list)
      error_mask = torch.tensor(package_error_flags).to(self.device)

      return final_tokens, error_mask

  @torch.no_grad()
  def run_simulation_step(self, original_image, image_caption, snr_db):
      """運行單一步驟模擬，並返回原始圖像、重建圖像和評估指標"""

      # 0. 數據轉換
      img_tensor_norm = transforms.ToTensor()(original_image).float().unsqueeze(0).to(self.device)

      # 1. 文本 Embedding
      text_embeddings = self.get_clip_embedding(image_caption)

      # 2. 圖像 Tokenization (TA-TiTok)
      if hasattr(self, 'tatitok'):
          # 實際 TA-TiTok 邏輯
          token_indices = self.tatitok.encode(img_tensor_norm)['indices'][0]
      else:
          # Mock
          token_indices = torch.randint(0, self.vocab_size, (self.tokens_per_image,)).to(self.device)

      # 3. 無線傳輸
      received_tokens_masked, error_mask = self.transmission_channel(token_indices, snr_db)

      # 4. Token 重建 (MaskGen)
      if hasattr(self, 'maskgen'):
          # 實際 MaskGen 邏輯
          # MaskGen.sample 需要知道要預測多少 token，通常是 12 次迭代
          reconstructed_tokens = self.maskgen.sample(received_tokens_masked, text_embeddings, num_steps=12)
      else:
          # Mock
          reconstructed_tokens = received_tokens_masked.clone()

      # 5. De-tokenization (TA-TiTok)
      if hasattr(self, 'tatitok'):
          # 實際 TA-TiTok decode 邏輯
          recon_image_tensor = self.tatitok.decode(reconstructed_tokens.unsqueeze(0), text_embeddings).clamp(0, 1)
      else:
          # Mock
          recon_image_tensor = transforms.GaussianBlur(5)(img_tensor_norm)


      # 評估指標
      if hasattr(self, 'lpips'):
          # 實際計算
          psnr = psnr_metric(img_tensor_norm, recon_image_tensor, data_range=1.0).item()
          lpips = self.lpips(img_tensor_norm, recon_image_tensor).item()
          clip_score = self.clip_metric(recon_image_tensor, image_caption).item() / 100.0 # 轉換為 0-1 範圍
      else:
          # Mocking Metric Results for demonstration
          per = error_mask.float().mean().item()
          psnr = 25 - snr_db * 0.5
          lpips = 0.1 + snr_db * 0.02
          clip_score = 0.8 - snr_db * 0.01

      return {
          'per': error_mask.float().mean().item(),
          'psnr': psnr,
          'lpips': lpips,
          'clip': clip_score,
          'original_img': original_image,
          'recon_img_tensor': recon_image_tensor
      }

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
sim_system = WirelessImageTransmissionSystem(device=DEVICE, drive_root=DRIVE_ROOT)
snr_range_db = np.arange(-5.0, 5.1, 1.0)
print(f"從 {DRIVE_ROOT} 載入 ImageNet 模擬數據集...")
dataset = ImageNetLoader(root_dir=os.path.join(DRIVE_ROOT, 'imagenet_val'), size=5)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

results = {}

for snr_db in snr_range_db:
  print(f"\n--- 模擬 SNR: {snr_db:.1f} dB ---")

  snr_results = {
      'psnr_list': [],
      'lpips_list': [],
      'clip_list': [],
      'per_list': []
  }

  for idx, (image, caption) in enumerate(dataloader):
      if idx >= 5: break # 僅處理前 5 張圖片以加快模擬速度

      image_pil = image[0]
      caption_text = caption[0]

      # 1. 產生描述
      final_caption = sim_system.generate_caption(image_pil)

      # 2. 執行單步模擬
      step_result = sim_system.run_simulation_step(image_pil, final_caption, snr_db)

      print(f"  Img {idx}: PER={step_result['per']:.2%}, PSNR={step_result['psnr']:.2f}, CLIP={step_result['clip']:.4f}")

      # 收集結果
      snr_results['psnr_list'].append(step_result['psnr'])
      snr_results['lpips_list'].append(step_result['lpips'])
      snr_results['clip_list'].append(step_result['clip'])
      snr_results['per_list'].append(step_result['per'])

  # 計算平均值
  results[snr_db] = {
      'avg_psnr': np.mean(snr_results['psnr_list']),
      'avg_lpips': np.mean(snr_results['lpips_list']),
      'avg_clip': np.mean(snr_results['clip_list']),
      'avg_per': np.mean(snr_results['per_list']),
  }


  print("\n--- 模擬結果彙總 (Average over samples) ---")
  print("SNR(dB) | Avg_PER | Avg_PSNR | Avg_LPIPS | Avg_CLIP")
  print("-----------------------------------------------------")
for snr, res in results.items():
  print(f"{snr:7.1f} | {res['avg_per']:.4f} | {res['avg_psnr']:.4f} | {res['avg_lpips']:.4f} | {res['avg_clip']:.4f}")